In [1]:
import simpy
import random
import numpy as np
import pandas as pd

# Parámetros globales
TIEMPO_SIMULACION = 8 * 60  # 8 horas en minutos
REPLICAS = 10

# Definición de tipos de usuarios según Tabla 1 y Tabla 2
# Formato: [Probabilidad, Media_Servicio, Media_Llegada]
DATOS_RETIRO = {
    'Rápido': [0.23, 1.0, 1.0],
    'Normal': [0.40, 2.0, 2.0],
    'Lento': [0.17, 3.0, 3.0],
    'Muy lento': [0.20, 4.0, 3.0]
}

DATOS_PAGO = {
    'Rápido': [0.10, 3.0, 1.0],
    'Normal': [0.20, 3.0, 2.0],
    'Lento': [0.30, 5.0, 3.0],
    'Muy lento': [0.40, 7.0, 4.0]
}

class BancoSimulacion:
    def __init__(self, env, num_cajeros_retiro, num_cajeros_pago):
        self.env = env
        self.cajeros_retiro = simpy.Resource(env, capacity=num_cajeros_retiro)
        self.cajeros_pago = simpy.Resource(env, capacity=num_cajeros_pago)
        self.estadisticas = {
            'tiempos_espera_retiro': [],
            'tiempos_espera_pago': [],
            'tiempos_atencion_retiro': [],
            'tiempos_atencion_pago': [],
            'usuarios_por_tipo': {
                'Retiro': {'Rápido': 0, 'Normal': 0, 'Lento': 0, 'Muy lento': 0},
                'Pago': {'Rápido': 0, 'Normal': 0, 'Lento': 0, 'Muy lento': 0}
            }
        }

    def atencion(self, tipo_accion, tipo_usuario, tiempo_servicio):
        yield self.env.timeout(random.expovariate(1.0 / tiempo_servicio))

    def llegada_usuario(self, tipo_accion, tipo_usuario, tiempo_servicio):
        llegada = self.env.now
        
        # Determinar a qué cola va el usuario
        if tipo_accion == 'Retiro':
            recurso = self.cajeros_retiro
            self.estadisticas['usuarios_por_tipo']['Retiro'][tipo_usuario] += 1
        else:
            recurso = self.cajeros_pago
            self.estadisticas['usuarios_por_tipo']['Pago'][tipo_usuario] += 1

        with recurso.request() as req:
            yield req
            espera = self.env.now - llegada
            
            # Registrar tiempos de espera
            if tipo_accion == 'Retiro':
                self.estadisticas['tiempos_espera_retiro'].append(espera)
            else:
                self.estadisticas['tiempos_espera_pago'].append(espera)
                
            # Proceso de atención
            tiempo_real_servicio = random.expovariate(1.0 / tiempo_servicio)
            yield self.env.timeout(tiempo_real_servicio)
            
            # Registrar tiempos de atención
            if tipo_accion == 'Retiro':
                self.estadisticas['tiempos_atencion_retiro'].append(tiempo_real_servicio)
            else:
                self.estadisticas['tiempos_atencion_pago'].append(tiempo_real_servicio)

def generador_usuarios(env, banco, tipo_accion, datos):
    """Genera usuarios de forma independiente para cada sub-tipo basándose en su tasa de llegada."""
    for tipo_usuario, params in datos.items():
        env.process(proceso_llegadas_especifico(env, banco, tipo_accion, tipo_usuario, params[2], params[1]))

def proceso_llegadas_especifico(env, banco, tipo_accion, tipo_usuario, media_llegada, media_servicio):
    while True:
        yield env.timeout(random.expovariate(1.0 / media_llegada))
        env.process(banco.llegada_usuario(tipo_accion, tipo_usuario, media_servicio))

def ejecutar_replica(num_cajeros_retiro, num_cajeros_pago):
    env = simpy.Environment()
    banco = BancoSimulacion(env, num_cajeros_retiro, num_cajeros_pago)
    
    # Iniciar generadores independientes
    generador_usuarios(env, banco, 'Retiro', DATOS_RETIRO)
    generador_usuarios(env, banco, 'Pago', DATOS_PAGO)
    
    env.run(until=TIEMPO_SIMULACION)
    return banco.estadisticas

def analizar_escenario(num_cajeros_retiro, num_cajeros_pago, nombre_escenario):
    resultados_replicas = []
    
    for i in range(REPLICAS):
        stats = ejecutar_replica(num_cajeros_retiro, num_cajeros_pago)
        resultados_replicas.append(stats)
        
    # Agregación de datos
    espera_retiro_avg = np.mean([np.mean(r['tiempos_espera_retiro']) if r['tiempos_espera_retiro'] else 0 for r in resultados_replicas])
    espera_pago_avg = np.mean([np.mean(r['tiempos_espera_pago']) if r['tiempos_espera_pago'] else 0 for r in resultados_replicas])
    
    atencion_retiro_avg = np.mean([np.mean(r['tiempos_atencion_retiro']) if r['tiempos_atencion_retiro'] else 0 for r in resultados_replicas])
    atencion_pago_avg = np.mean([np.mean(r['tiempos_atencion_pago']) if r['tiempos_atencion_pago'] else 0 for r in resultados_replicas])
    
    # Totales de usuarios por tipo en promedio por réplica
    usuarios_totales = {'Retiro': {}, 'Pago': {}}
    for tipo_accion in ['Retiro', 'Pago']:
        for subtipo in DATOS_RETIRO.keys():
            avg_users = np.mean([r['usuarios_por_tipo'][tipo_accion][subtipo] for r in resultados_replicas])
            usuarios_totales[tipo_accion][subtipo] = avg_users

    print(f"\n--- Resultados: {nombre_escenario} ---")
    print(f"Tiempo promedio de espera - Retiros: {espera_retiro_avg:.2f} min | Pagos/Consignaciones: {espera_pago_avg:.2f} min")
    print(f"Tiempo promedio de atención - Retiros: {atencion_retiro_avg:.2f} min | Pagos/Consignaciones: {atencion_pago_avg:.2f} min")
    print(f"Promedio de usuarios atendidos por réplica (8h):")
    print(pd.DataFrame(usuarios_totales))

if __name__ == "__main__":
    print("Ejecutando Modelo 1: 2 Cajeros Retiro, 1 Cajero Pago")
    analizar_escenario(2, 1, "Modelo 1 (2 Retiros, 1 Pago)")
    
    print("\nEjecutando Modelo 2: 1 Cajero Retiro, 2 Cajeros Pago")
    analizar_escenario(1, 2, "Modelo 2 (1 Retiro, 2 Pagos)")

Ejecutando Modelo 1: 2 Cajeros Retiro, 1 Cajero Pago

--- Resultados: Modelo 1 (2 Retiros, 1 Pago) ---
Tiempo promedio de espera - Retiros: 128.96 min | Pagos/Consignaciones: 206.58 min
Tiempo promedio de atención - Retiros: 1.97 min | Pagos/Consignaciones: 3.89 min
Promedio de usuarios atendidos por réplica (8h):
           Retiro   Pago
Rápido      488.0  491.9
Normal      238.5  245.3
Lento       161.2  157.3
Muy lento   161.9  121.3

Ejecutando Modelo 2: 1 Cajero Retiro, 2 Cajeros Pago

--- Resultados: Modelo 2 (1 Retiro, 2 Pagos) ---
Tiempo promedio de espera - Retiros: 186.41 min | Pagos/Consignaciones: 172.02 min
Tiempo promedio de atención - Retiros: 1.96 min | Pagos/Consignaciones: 3.67 min
Promedio de usuarios atendidos por réplica (8h):
           Retiro   Pago
Rápido      476.2  487.1
Normal      241.8  236.3
Lento       161.1  159.7
Muy lento   164.6  118.7
